In [48]:
import pandas as pd
import numpy as np
import tensorflow as tf
from agent import EnhancedDQNAgent  # Use our improved agent
import random
from cnn_dqn_model import CNNPlacementQNetwork
from product_placement_env import ProductPlacementEnv
import matplotlib.pyplot as plt
import os

def get_data():
    """Load and preprocess product data with enhanced normalization"""
    file_path = 'productos_anaquel.xlsx'
    df_list = []
    i = 1
    try:
        while True:
            df_list.append(pd.read_excel(file_path, sheet_name=f"Sheet {i}"))
            i += 1
    except Exception as e:
        pass

    df_all = pd.concat(df_list, ignore_index=True)
    df_all = df_all[df_all['ANAQUEL'].str.startswith('C', na=False)]
    #print(df_all.head())
    # Data cleaning and normalization
    df_all['UNDESTIMADAS'] = df_all['UNDESTIMADAS'].apply(lambda x: max(x, 1))  # Ensure positive
    df_all = df_all.drop_duplicates(subset='PRODUCTO')
    exclude_campa = [201915, 201916, 201917, 201918, 202002, 202003, 202005, 202006,
                     202008, 202009, 202011, 202010, 202012, 202004, 202007]
    df_all = df_all[~df_all['CAMPA'].isin(exclude_campa)]
    
    df_all = df_all[['PRODUCTO','ALTO', 'ANCHO', 'LARGO', 'VOLUMEN', 'PESO', "UNDESTIMADAS", "CAMPA", "ANAQUEL"]]
    
    
    # Robust normalization: use min-max but handle outliers
    for col in ['ALTO', 'ANCHO', 'LARGO', 'VOLUMEN', 'PESO', 'UNDESTIMADAS']:
        # Calculate percentiles to handle outliers
        q_low = df_all[col].quantile(0.1)
        q_high = df_all[col].quantile(0.8)
        
        # Clip values to reduce impact of outliers
        df_all[col] = df_all[col].clip(q_low, q_high)
        
        # Apply min-max normalization
        df_all[col] = (df_all[col] - df_all[col].min()) / (df_all[col].max() - df_all[col].min() + 1e-8)
    
    # print(df_all.describe())
    df_all.reset_index(drop=True, inplace=True)
    return df_all

In [49]:
df = get_data()

In [50]:
df

,PRODUCTO,ALTO,ANCHO,LARGO,VOLUMEN,PESO,UNDESTIMADAS,CAMPA,ANAQUEL
0,200063628,0.152778,0.228723,0.000000,0.000000,0.002588,1.0,201416,C02A4
1,200063636,1.000000,0.728723,0.247059,0.637566,0.758231,1.0,201416,C21C6
2,200063644,1.000000,0.707447,0.247059,0.620259,0.772189,1.0,201416,C27C1
3,200063815,0.361111,0.265957,0.000000,0.008361,0.027268,1.0,201416,C02B6
4,200063818,0.361111,0.265957,0.000000,0.008361,0.027638,1.0,201416,C01A1
...,...,...,...,...,...,...,...,...,...
12428,400101764,0.027778,0.877660,1.000000,0.419952,0.883663,0.0,202503,C05C7
12429,400101782,0.027778,0.877660,1.000000,0.419952,0.617825,0.0,202503,C13C7
12430,400101800,0.000000,0.877660,1.000000,0.195559,0.581406,0.0,202503,C19C1
12431,400101819,0.000000,0.627660,1.000000,0.075684,0.196883,0.0,202503,C11C1


In [51]:
df.describe()

,PRODUCTO,ALTO,ANCHO,LARGO,VOLUMEN,PESO,UNDESTIMADAS,CAMPA
count,1.243300e+04,12433.000000,12433.000000,12433.000000,12433.000000,12433.000000,12433.000000,12433.000000
mean,2.264204e+08,0.476159,0.479174,0.501073,0.349567,0.374926,0.360664,201980.750583
std,6.095819e+07,0.352712,0.365424,0.349821,0.384279,0.393921,0.396155,290.376358
min,2.000362e+08,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,201416.000000
25%,2.000998e+08,0.166667,0.170213,0.223529,0.022778,0.032814,0.000846,201717.000000
50%,2.100788e+08,0.444444,0.388298,0.441176,0.162012,0.197160,0.198715,201912.000000
75%,2.100983e+08,0.861111,0.867021,0.870588,0.656020,0.788734,0.734822,202213.000000
max,4.001025e+08,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,202503.000000


In [58]:
# Extract the numeric part after 'D' in 'ANAQUEL' (e.g., C30D1 -> 1)
aux = df.copy()
aux['zona'] = df['ANAQUEL'].str.extract(r'C(\d+)').astype(int)
# Extrae la letra entre los dígitos y la convierte a número: A=1, B=2, C=3, etc.
aux['fila'] = aux['ANAQUEL'].str.extract(r'\d+([A-Z])\d+')[0].map(lambda x: ord(x) - ord('A') + 1 if pd.notnull(x) else np.nan).astype('Int64')
# Extrae el último dígito como columna
aux['columna'] = aux['ANAQUEL'].str.extract(r'(\d+)$').astype(int)

aux


,PRODUCTO,ALTO,ANCHO,LARGO,VOLUMEN,PESO,UNDESTIMADAS,CAMPA,ANAQUEL,zona,fila,columna
0,200063628,0.152778,0.228723,0.000000,0.000000,0.002588,1.0,201416,C02A4,2,1,4
1,200063636,1.000000,0.728723,0.247059,0.637566,0.758231,1.0,201416,C21C6,21,3,6
2,200063644,1.000000,0.707447,0.247059,0.620259,0.772189,1.0,201416,C27C1,27,3,1
3,200063815,0.361111,0.265957,0.000000,0.008361,0.027268,1.0,201416,C02B6,2,2,6
4,200063818,0.361111,0.265957,0.000000,0.008361,0.027638,1.0,201416,C01A1,1,1,1
...,...,...,...,...,...,...,...,...,...,...,...,...
12428,400101764,0.027778,0.877660,1.000000,0.419952,0.883663,0.0,202503,C05C7,5,3,7
12429,400101782,0.027778,0.877660,1.000000,0.419952,0.617825,0.0,202503,C13C7,13,3,7
12430,400101800,0.000000,0.877660,1.000000,0.195559,0.581406,0.0,202503,C19C1,19,3,1
12431,400101819,0.000000,0.627660,1.000000,0.075684,0.196883,0.0,202503,C11C1,11,3,1


In [83]:

# Selecciona una CAMPA específica, por ejemplo la primera disponible
campa_elegida = aux['CAMPA'].iloc[0]
df_campa = aux[aux['CAMPA'] == campa_elegida]
df_campa = df_campa[['PRODUCTO', 'zona', 'fila', 'columna']].reset_index(drop=True)
print(f"Seleccionando campaña: {campa_elegida} con {len(df_campa['PRODUCTO'].unique())} productos únicos.")

import numpy as np
# Crea la matriz vacía
matriz = np.zeros((7, 126), dtype=int)

for i, row in df_campa.iterrows():
    print(i)
    zona = row['zona']
    fila = row['fila']
    columna = row['columna']
    producto = row['PRODUCTO']
    #print(f"Colocando producto {producto} en zona {zona}, fila {fila}, columna {columna}")
    if zona%2 == 0:
        fila_real = fila + 3     # filas 1 a 3 (índice 1-3 en python es 0-2)
        columna_real =  columna + (((zona//2) - 1) * 7)
    else:
        fila_real = fila  # filas 4 a 7 (índice 4-7 en python es 3-6)
        columna_real =  columna + ((zona//2) * 7)
    #print(f"Colocando en fila {fila_real} y columna {columna_real}")
    if matriz[fila_real-1, columna_real-1] != 0:
        print(f"Advertencia: Posición ocupada en fila {fila_real}, columna {columna_real}. Reemplazando producto {matriz[fila_real-1, columna_real-1]} con {producto}.")
    matriz[fila_real-1, columna_real-1] = producto  # Ajusta el índice para que empiece en 0


Seleccionando campaña: 201416 con 421 productos únicos.
0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49
50
51
52
53
54
55
56
57
58
59
60
61
62
63
64
65
66
67
68
69
70
71
72
73
74
75
76
77
78
79
80
81
82
83
84
85
86
87
88
89
90
91
92
93
94
95
96
97
98
99
100
101
102
103
104
105
106
107
108
109
110
111
112
113
114
115
116
117
118
119
120
121
122
123
124
125
126
127
128
129
130
131
132
133
134
135
136
137
138
139
140
141
142
143
144
145
146
147
148
149
150
151
152
153
154
155
156
157
158
159
160
161
162
163
164
165
166
167
168
169
170
171
172
173
174
175
176
177
178
179
180
181
182
183
184
185
186
187
188
189
190
191
192
193
194
195
196
197
198
199
200
201
202
203
204
205
206
207
208
209
210
211
212
213
214
215
216
217
218
219
220
221
222
223
224
225
226
227
228
229
230
231
232
233
234
235
236
237
238
239
240
241
242
243
244
245
246
247
248
249
250
251
252
253
254
255
256
257
258
259
260
261
262
26

In [84]:
import pandas as pd

# Print the matriz in a pretty format (terminal style, no DataFrame)

for i, row in enumerate(matriz):
    pretty_row = ["{:10}".format(str(x) if x != 0 else "") for x in row]
    print(" | ".join(pretty_row))

total_items = np.count_nonzero(matriz)
print(f"Total items (non-zero): {total_items}")


200063818  |            |            |            |            | 200077458  | 200058348  | 210077298  | 210077229  | 210071599  | 210074930  | 210066194  | 210075959  |            | 210074200  |            | 200058335  | 210077408  | 200072491  | 200058474  | 200058331  |            | 210077671  | 200072583  |            | 210063787  | 210077376  | 210069605  |            | 210072889  | 200079542  |            |            | 210077323  | 200058327  |            | 210077226  | 200079631  | 210077666  |            | 200076780  | 210071089  |            | 200064538  | 210065182  | 200076964  |            | 200079424  | 210077846  | 210074519  | 200079399  | 210077351  | 200064912  |            | 200080792  | 200076963  | 210077398  |            | 200072741  | 200077461  | 200063042  | 210077381  | 210077407  | 200079724  | 210078085  | 200073387  | 210077299  |            | 210078398  | 200058332  |            | 200064539  |            | 200079697  |            | 200064317  | 210068091  |